# Fine-tune a pretrained language model with TrainLM

Use this when you already have pretrained weights. TrainLM loads the model and tokenizer, prepares a plain-text dataset, fine-tunes, evaluates, and saves the model locally. This is different from pretraining from scratch, which starts with random weights.

Run the cells from top to bottom. The defaults use a tiny GPT-2 model and three steps as a workflow smoke test; they do not measure model quality. Change the model and training budget in the settings cell for a real run. Nothing is uploaded to the Hub.

Native TorchTPU is experimental. Select it explicitly in the settings cell; this notebook does not claim TPU training has been certified.

In [ ]:
#@title 1. Install TrainLM without replacing Torch
from importlib import metadata, util
import subprocess
import sys

# This notebook is on the same feature branch as the API. After it merges,
# change this to "main".
TRAINLM_REF = "feat/01-pretrained-finetuning-api"
trainlm_source = (
    "git+https://github.com/Dhiraj309/TrainLM.git@" + TRAINLM_REF
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "--no-deps", trainlm_source,
    ],
    check=True,
)

# Install only the optional dataset package if the runtime does not have it.
if util.find_spec("datasets") is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--quiet",
            "--disable-pip-version-check", "datasets>=3,<5",
        ],
        check=True,
    )

import torch
import transformers
import datasets
print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"datasets={metadata.version('datasets')}")

## 2. Choose the model, dataset, and run size

The dataset revision must be an immutable 40-character commit SHA, and the dataset must have separate training and validation splits. The example is intentionally small. For useful training, choose a model that fits your runtime and increase `MAX_STEPS` after the smoke run succeeds.

For native TorchTPU, select `native_tpu` and use a TPU runtime. This path is experimental; `auto` does not select TPU yet.

In [ ]:
#@title 3. Edit these run settings
from pathlib import Path

from trainlm import (
    TrainLMFineTuningConfig,
    TrainLMTrainingArguments,
    run_finetuning,
)

MODEL_NAME_OR_PATH = "sshleifer/tiny-gpt2" #@param {type:"string"}
MODEL_REVISION = "" #@param {type:"string"}
DATASET_ID = "roneneldan/TinyStories" #@param {type:"string"}
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64" #@param {type:"string"}
TEXT_FIELD = "text" #@param {type:"string"}
TRAIN_SPLIT = "train" #@param {type:"string"}
VALIDATION_SPLIT = "validation" #@param {type:"string"}
MAX_STEPS = 3 #@param {type:"integer"}
SEQUENCE_LENGTH = 128 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
LEARNING_RATE = 2e-5 #@param {type:"number"}
ACCELERATOR = "cpu" #@param ["cpu", "cuda", "native_tpu"]
OUTPUT_DIR = Path("runs/trainlm-finetuning")

config = TrainLMFineTuningConfig(
    model_name_or_path=MODEL_NAME_OR_PATH,
    model_revision=MODEL_REVISION or None,
    dataset_id=DATASET_ID,
    dataset_revision=DATASET_REVISION,
    text_field=TEXT_FIELD,
    train_split=TRAIN_SPLIT,
    validation_split=VALIDATION_SPLIT,
    args=TrainLMTrainingArguments(
        output_dir=OUTPUT_DIR,
        max_steps=MAX_STEPS,
        sequence_length=SEQUENCE_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        accelerator=ACCELERATOR,
        bf16=ACCELERATOR == "native_tpu",
    ),
)
print(f"model={MODEL_NAME_OR_PATH}")
print(f"dataset={DATASET_ID}@{DATASET_REVISION}")
print(f"steps={MAX_STEPS}, sequence_length={SEQUENCE_LENGTH}, accelerator={ACCELERATOR}")

## 3. Fine-tune and evaluate

TrainLM keeps the training loop and padding-label handling inside the API. This call evaluates on the configured validation split and writes the model and tokenizer to `OUTPUT_DIR / "model"`.

In [ ]:
result = run_finetuning(config)
print(result.summary())

## 4. Download the model

TrainLM writes a JSON run summary beside the model export with the settings, resolved revisions, and metrics. The bundle contains the exported model, tokenizer, and that summary. Kaggle users can retrieve it from the notebook's output files; Colab starts a direct download.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

summary_path = result.summary_path
if summary_path is None or not summary_path.is_file():
    raise RuntimeError("TrainLM did not write the fine-tuning summary.")
bundle_path = OUTPUT_DIR / "finetuned-model.zip"
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for path in result.model_path.rglob("*"):
        if path.is_file():
            bundle.write(path, path.relative_to(result.model_path.parent))
    bundle.write(summary_path, summary_path.name)

try:
    from google.colab import files
except ImportError:
    print(f"Kaggle or local runtime: download {bundle_path}")
else:
    files.download(str(bundle_path))